In [103]:
import pandas as pd
from scipy import stats
import numpy as np


#df = pd.read_csv('D:\\Projetos\\AB test analysis\\Fast Food Marketing Campaign AB Test\\WA_Marketing-Campaign.csv')
df = pd.read_csv('D:\\Projetos\\AB test analysis\\Fast Food Marketing Campaign AB Test\\WA_Marketing-Campaign.csv')#.query('MarketSize == "Small"')#.query('MarketID != 2')

def check_bias_and_stratification(df, group_col='Promotion', covariates=['MarketSize', 'AgeOfStore', 'LocationID', 'MarketID']):
    """
    Verifica se há viés prévio nos grupos do teste A/B ou se a estratificação foi adequada.
    
    Retorna um dicionário com:
    - 'summary': Dicionário com DataFrames para numérico e categórico
    - 'tests': Dicionário com resultados de testes de diferença entre grupos
    - 'bias_report': Relatório textual sobre possíveis vieses
    """
    results = {}
    
    # Separar covariáveis numéricas e categóricas
    numerical_covs = [cov for cov in covariates if df[cov].dtype in ['int64', 'float64']]
    categorical_covs = [cov for cov in covariates if cov not in numerical_covs]
    
    # Estatísticas descritivas
    summary_num = df.groupby(group_col)[numerical_covs].agg(['mean', 'std', 'count']) if numerical_covs else pd.DataFrame()
    summary_cat = {}
    for cov in categorical_covs:
        summary_cat[cov] = df.groupby(group_col)[cov].value_counts().unstack().fillna(0)
    
    results['summary'] = {'numerical': summary_num, 'categorical': summary_cat}
    
    # Testes de diferença
    tests = {}
    for cov in covariates:
        groups = [df[df[group_col] == g][cov] for g in df[group_col].unique()]
        
        if cov in categorical_covs:
            # Chi-square test
            contingency = pd.crosstab(df[group_col], df[cov])
            chi2, p, dof, expected = stats.chi2_contingency(contingency)
            tests[cov] = {'test': 'Chi-square', 'statistic': chi2, 'p_value': p}
        else:  # Numérico
            # ANOVA se mais de 2 grupos
            if len(groups) > 2:
                f_stat, p = stats.f_oneway(*groups)
                tests[cov] = {'test': 'ANOVA', 'statistic': f_stat, 'p_value': p}
            else:
                t_stat, p = stats.ttest_ind(*groups)
                tests[cov] = {'test': 't-test', 'statistic': t_stat, 'p_value': p}
    
    results['tests'] = tests
    
    # Relatório de viés
    bias_issues = []
    for cov, test in tests.items():
        if test['p_value'] < 0.05:
            bias_issues.append(f"Possível viés em {cov}: diferença significativa entre grupos (p={test['p_value']:.4f})")
    
    if bias_issues:
        results['bias_report'] = "Foram detectadas possíveis fontes de viés:\n" + "\n".join(bias_issues)
    else:
        results['bias_report'] = "Não foram detectadas diferenças significativas entre os grupos. A estratificação parece adequada."
    
    return results



In [104]:
# Analise de vies e estratificação
result = check_bias_and_stratification(df)
print("Resumo estatístico:")
print(result['summary'])
print("\nTestes de diferença:")
for cov, test in result['tests'].items():
    print(f"{cov}: {test['test']} - p-value = {test['p_value']:.4f}")
print("\nRelatório de viés:")
print(result['bias_report'])

Resumo estatístico:
{'numerical':           AgeOfStore                  LocationID                    MarketID  \
                mean       std count        mean         std count      mean   
Promotion                                                                      
1           8.279070  6.636160   172  488.465116  299.352389   172  5.790698   
2           7.978723  6.597648   188  497.446809  290.158047   188  5.893617   
3           9.234043  6.651646   188  453.808511  274.555052   188  5.468085   

                           
                std count  
Promotion                  
1          2.993624   172  
2          2.897419   188  
3          2.742816   188  , 'categorical': {'MarketSize': MarketSize  Large  Medium  Small
Promotion                       
1              56      96     20
2              64     108     16
3              48     116     24}}

Testes de diferença:
MarketSize: Chi-square - p-value = 0.3135
AgeOfStore: ANOVA - p-value = 0.1615
LocationID: ANOVA 

In [105]:
# Explorar os dados faltantes para MarketID == 2 e Promotion == 2
print("Dados gerais:")
print(f"Shape do dataframe: {df.shape}")
print(f"\nColunas: {df.columns.tolist()}")

# Verificar a distribuição de dados por MarketID e Promotion
print("\nDistribuição de registros por MarketID e Promotion:")
dist = df.groupby(['MarketID', 'Promotion']).size().unstack(fill_value=0)
print(dist)

# Verificar especificamente MarketID == 2
print("\nDados para MarketID == 2:")
market_2 = df[df['MarketID'] == 2]
print(f"Total de registros: {len(market_2)}")
print(f"Promoções disponíveis: {sorted(market_2['Promotion'].unique())}")
print(f"Registros por promoção em MarketID == 2:")
print(market_2['Promotion'].value_counts().sort_index())

# Verificar o tamanho do mercado 2
print(f"\nMarketSize do MarketID == 2: {market_2['MarketSize'].unique()}")


Dados gerais:
Shape do dataframe: (548, 7)

Colunas: ['MarketID', 'MarketSize', 'LocationID', 'AgeOfStore', 'Promotion', 'week', 'SalesInThousands']

Distribuição de registros por MarketID e Promotion:
Promotion   1   2   3
MarketID             
1          20  20  12
2           4   0  20
3          28  24  36
4          16  16   4
5           8  32  20
6          20  24  16
7          16  16  28
8          20   8  20
9          12   8  20
10         28  40  12

Dados para MarketID == 2:
Total de registros: 24
Promoções disponíveis: [np.int64(1), np.int64(3)]
Registros por promoção em MarketID == 2:
Promotion
1     4
3    20
Name: count, dtype: int64

MarketSize do MarketID == 2: ['Small']


In [106]:
# Criar dados sintéticos usando regressão linear para MarketID == 2, Promotion == 2
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

def generate_synthetic_data_with_linear_regression(df, target_market_id=2, target_promotion=2, market_size_filter='Small'):
    """
    Gera dados sintéticos para uma combinação ausente de MarketID e Promotion
    usando regressão linear com normalização StandardScaler (z-score).
    
    Parâmetros:
    - df: DataFrame com os dados
    - target_market_id: ID do mercado para o qual gerar dados sintéticos
    - target_promotion: ID da promoção para a qual gerar dados sintéticos
    - market_size_filter: Tamanho do mercado para usar como base
    
    Retorna: DataFrame com os dados sintéticos gerados
    """
    
    # Filtrar dados dos mercados com o mesmo tamanho
    same_size_markets = df[df['MarketSize'] == market_size_filter].copy()
    
    # Obter informações do mercado alvo
    target_market_info = df[df['MarketID'] == target_market_id].iloc[0]
    target_age = target_market_info['AgeOfStore']
    target_location = target_market_info['LocationID']
    
    # Definir colunas numéricas a normalizar
    le_market_size = LabelEncoder()

    
    numeric_cols = ['week', 'Promotion', 'AgeOfStore', 'MarketID']
    
    # Criar e aplicar StandardScaler aos dados de treino
    scaler = StandardScaler()

    X_original = same_size_markets[numeric_cols].copy()
    X_scaled = scaler.fit_transform(X_original)
    X = pd.DataFrame(X_scaled, columns=numeric_cols)
    y = same_size_markets['SalesInThousands'].copy()
    
    # Treinar modelo de regressão linear com dados normalizados
    model = LinearRegression()
    model.fit(X, y)
    
    # Obter dados existentes do mercado 2 com promoções 1 e 3 para entender o padrão
    market_2_data = df[df['MarketID'] == target_market_id]
    print(f"Dados disponíveis para MarketID == {target_market_id}:")
    print(market_2_data[['MarketID', 'Promotion', 'week', 'AgeOfStore', 'LocationID', 'SalesInThousands']])
    
    # Criar dados sintéticos para cada semana que existe no mercado 2
    weeks_in_market = sorted(market_2_data['week'].unique())
    synthetic_rows = []
    
    print(f"\nGerando dados sintéticos para MarketID == {target_market_id}, Promotion == {target_promotion}:")
    
    for week in weeks_in_market:
        print(f"  Processando week={week}...")
        # Preparar features para predição e normalizar usando o scaler treinado
        X_pred_raw = np.array([[week, target_promotion, target_age, target_market_id]])
        X_pred_scaled = scaler.transform(X_pred_raw)
        
        # Fazer predição com dados normalizados
        predicted_sales = model.predict(X_pred_scaled)[0]
        
        # Criar nova linha com dados sintéticos
        synthetic_row = {
            'MarketID': target_market_id,
            'MarketSize': market_size_filter,
            'LocationID': target_location,
            'AgeOfStore': target_age,
            'Promotion': target_promotion,
            'week': week,
            'SalesInThousands': predicted_sales
        }
        synthetic_rows.append(synthetic_row)
        print(f"  week={week}: predicted SalesInThousands = {predicted_sales:.2f}")
    
    return pd.DataFrame(synthetic_rows)

# Gerar dados sintéticos
synthetic_data = generate_synthetic_data_with_linear_regression(df, target_market_id=2, target_promotion=2, market_size_filter='Small')

print(f"\nDados sintéticos gerados:")
print(synthetic_data)


Dados disponíveis para MarketID == 2:
    MarketID  Promotion  week  AgeOfStore  LocationID  SalesInThousands
52         2          1     1          22         101             67.48
53         2          1     2          22         101             65.57
54         2          1     3          22         101             68.42
55         2          1     4          22         101             60.93
56         2          3     1           8         102             61.59
57         2          3     2           8         102             63.64
58         2          3     3           8         102             54.68
59         2          3     4           8         102             61.24
60         2          3     1          22         103             62.93
61         2          3     2          22         103             58.77
62         2          3     3          22         103             70.60
63         2          3     4          22         103             65.06
64         2          3   

c:\Users\oguis\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\oguis\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\oguis\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\oguis\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\oguis\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:269

In [107]:
# Adicionar dados sintéticos ao dataframe original
df_augmented = pd.concat([df, synthetic_data], ignore_index=True)

df_augmented.to_csv('D:\\Projetos\\AB test analysis\\Fast Food Marketing Campaign AB Test\\WA_Marketing-Campaign-Ajustado.csv', index=False)
print(f"DataFrame original: {df.shape[0]} registros")
print(f"DataFrame após adição dos dados sintéticos: {df_augmented.shape[0]} registros")
print(f"Registros adicionados: {len(synthetic_data)}")

# Verificar a nova distribuição
print("\nNova distribuição de registros por MarketID e Promotion:")
new_dist = df_augmented.groupby(['MarketID', 'Promotion']).size().unstack(fill_value=0)
print(new_dist)

# Verificar especificamente MarketID == 2
print("\nDados atualizados para MarketID == 2:")
market_2_updated = df_augmented[df_augmented['MarketID'] == 2]
print(f"Total de registros: {len(market_2_updated)}")
print(f"Promoções disponíveis: {sorted(market_2_updated['Promotion'].unique())}")
print(f"Registros por promoção em MarketID == 2:")
print(market_2_updated['Promotion'].value_counts().sort_index())

print("\nAmostra dos dados sintéticos adicionados:")
print(df_augmented[df_augmented['Promotion'] == 2].head())



DataFrame original: 548 registros
DataFrame após adição dos dados sintéticos: 552 registros
Registros adicionados: 4

Nova distribuição de registros por MarketID e Promotion:
Promotion   1   2   3
MarketID             
1          20  20  12
2           4   4  20
3          28  24  36
4          16  16   4
5           8  32  20
6          20  24  16
7          16  16  28
8          20   8  20
9          12   8  20
10         28  40  12

Dados atualizados para MarketID == 2:
Total de registros: 28
Promoções disponíveis: [np.int64(1), np.int64(2), np.int64(3)]
Registros por promoção em MarketID == 2:
Promotion
1     4
2     4
3    20
Name: count, dtype: int64

Amostra dos dados sintéticos adicionados:
    MarketID MarketSize  LocationID  AgeOfStore  Promotion  week  \
4          1     Medium           2           5          2     1   
5          1     Medium           2           5          2     2   
6          1     Medium           2           5          2     3   
7          1     Med

In [108]:
# Resumo: Dados sintéticos criados para MarketID == 2 e Promotion == 2
print("=" * 80)
print("RESUMO DA GERAÇÃO DE DADOS SINTÉTICOS")
print("=" * 80)
print(f"\nMétodo: Regressão Linear")
print(f"Base de treino: Mercados com MarketSize == 'Small'")
print(f"Mercado alvo: MarketID == 2 (Small)")
print(f"Promoção alvo: Promotion == 2")
print(f"\nResultados:")
print(f"  - {len(synthetic_data)} registros sintéticos foram gerados (um por semana)")
print(f"  - Dados adicionados ao DataFrame: df_augmented")
print(f"\nDistribuição final para MarketID == 2:")
print(f"  - Promotion 1: 4 registros")
print(f"  - Promotion 2: 4 registros (SINTÉTICOS)")
print(f"  - Promotion 3: 20 registros")

print(f"\nDetalhes dos dados sintéticos gerados:")
print(synthetic_data.to_string())

print(f"\n{'AVISO:'} Os dados sintéticos em df_augmented podem ser usados nas análises subsequentes.")
print(f"Se desejar manter análises separadas, use:")
print(f"  - df: DataFrame original (548 registros)")
print(f"  - df_augmented: DataFrame com dados sintéticos (552 registros)")


RESUMO DA GERAÇÃO DE DADOS SINTÉTICOS

Método: Regressão Linear
Base de treino: Mercados com MarketSize == 'Small'
Mercado alvo: MarketID == 2 (Small)
Promoção alvo: Promotion == 2

Resultados:
  - 4 registros sintéticos foram gerados (um por semana)
  - Dados adicionados ao DataFrame: df_augmented

Distribuição final para MarketID == 2:
  - Promotion 1: 4 registros
  - Promotion 2: 4 registros (SINTÉTICOS)
  - Promotion 3: 20 registros

Detalhes dos dados sintéticos gerados:
   MarketID MarketSize  LocationID  AgeOfStore  Promotion  week  SalesInThousands
0         2      Small         101          22          2     1         63.907381
1         2      Small         101          22          2     2         64.511514
2         2      Small         101          22          2     3         65.115647
3         2      Small         101          22          2     4         65.719781

AVISO: Os dados sintéticos em df_augmented podem ser usados nas análises subsequentes.
Se desejar manter aná